In [3]:
import pandas as pd
import numpy as np
import json
import glob
import csv
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

import spacy
from nltk.corpus import stopwords
import pyLDAvis
import pyLDAvis.gensim
import pyLDAvis.gensim_models

/var/folders/d9/sbfhfygx5vx9mlfxxhcxrb280000gn/T/ipykernel_10515/2475999428.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [4]:
def load_data(file):
    with open (file, "r", encoding="utf-8") as f:
        data = json.load(f)
    return (data)

def write_data(file, data):
    with open (file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [5]:
stopwords = stopwords.words("english")
stopwords.append("be")
print(stopwords)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [6]:
# Load the JSON data from the file
with open("vir_bag_of_words.json", "r") as file:
    data = json.load(file)

# Get the keys of the JSON object
fields = data.keys()

# Print the fields
print(fields)


dict_keys(['BagOfWords'])


In [7]:
data = load_data("vir_bag_of_words.json")['BagOfWords']
print (data[0][0:90])
# print (data[1][0:90])

i,lost,80%,of,my,mind,it,is,very,freeing,you,should,see,the,look,on,your,faces,right,now,b


In [8]:
def lemmatization(texts, allowed_postages=["NOUN", "ADJ", "VERB", "ADV"]):
    nlp = nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    texts_out = []
    for text in data:
        doc = nlp(text)
        new_text = []
        for token in doc:
            if token.pos_ in allowed_postages:
                new_text.append(token.lemma_)
        final = " ".join(new_text)
        texts_out.append(final)
    return (texts_out)

lemmatized_texts = lemmatization(data)
print (lemmatized_texts[0][0:150])

mind very free see look on face now way good guy excited name such good time tonight so excited delightful talk now just think time really embrace roo


In [9]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download()

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


True

In [10]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'look', 'write', 'make', 'shit', 'take', 'tell', 'subject', 're', 'edu', 'use', 'be', 'know', 'go', 'think', 'come', 'see', 'guy', 'say', 'even', 'year', 'one', 'would', 'find', 'get'])
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/alisha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
def gen_words(data):
    final = []
    for text in data:
        new = gensim.utils.simple_preprocess(text, deacc=True)
        final.append(new)
    return (final)

data_words = gen_words(lemmatized_texts)

print (data_words[0][0:20])
# print (data_words[1][0:20])

['mind', 'very', 'free', 'see', 'look', 'on', 'face', 'now', 'way', 'good', 'guy', 'excited', 'name', 'such', 'good', 'time', 'tonight', 'so', 'excited', 'delightful']


In [12]:
data_words = remove_stopwords(data_words)
print(data_words[:1][0][:30])

['mind', 'free', 'face', 'way', 'good', 'excited', 'name', 'good', 'time', 'tonight', 'excited', 'delightful', 'talk', 'time', 'really', 'embrace', 'root', 'comedy', 'authentically', 'indian', 'really', 'indian', 'fake', 'american', 'accent', 'understand', 'opportunity', 'history', 'tonight', 'first']


In [13]:
#BIGRAMS AND TRIGRAMS
bigram_phrases = gensim.models.Phrases(data_words, min_count=5, threshold=200)
trigram_phrases = gensim.models.Phrases(bigram_phrases[data_words], threshold=100)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
    return([bigram[doc] for doc in texts])

def make_trigrams(texts):
    return ([trigram[bigram[doc]] for doc in texts])

data_bigrams = make_bigrams(data_words)
data_bigrams_trigrams = make_trigrams(data_bigrams)

print (data_bigrams_trigrams[0])

['mind', 'free', 'face', 'way', 'good', 'excited', 'name', 'good', 'time', 'tonight', 'excited', 'delightful', 'talk', 'time', 'really', 'embrace', 'root', 'comedy', 'authentically', 'indian', 'really', 'indian', 'fake', 'american', 'accent', 'understand', 'opportunity', 'history', 'tonight', 'first', 'ever', 'indian', 'leave', 'never', 'happen', 'stick', 'around', 'kick', 'news', 'week', 'work', 'leave', 'browner', 'pasture', 'honestly', 'honestly', 'government', 'ban', 'beef', 'like', 'international', 'career', 'bad', 'beef', 'good', 'couple', 'first', 'world', 'tour', 'entire', 'world', 'countrie', 'world', 'common', 'thing', 'number', 'masturbate', 'country', 'thank', 'holiday', 'chain', 'everywhere', 'hotel', 'memory', 'foam', 'memory', 'matter', 'entire', 'world', 'people', 'thing', 'love', 'indian', 'people', 'smart', 'indian', 'people', 'lead', 'believe', 'rest', 'world', 'smart', 'answer', 'question', 'whenever', 'says', 'indian', 'people', 'smart', 'reality', 'smart', 'percen

In [14]:
# TF-IDF REMOVAL
from gensim.models import TfidfModel

id2word = corpora.Dictionary(data_bigrams_trigrams)

texts = data_bigrams_trigrams

corpus = [id2word.doc2bow(text) for text in texts]
print (corpus[0][0:20])

tfidf = TfidfModel(corpus, id2word=id2word)

low_value = 0.03
words = []
words_missing_in_tfidf = []
for i in range (0, len(corpus)):
    bow = corpus[i]
    low_value_words = [] # reinitialization to be safe, you can skip this
    tfidf_ids = [id for id, value in bow]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id, value in tfidf[bow] if value < low_value]
    drops = low_value_words+words_missing_in_tfidf
    for item in drops:
        words.append(id2word[item])
    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # the words with tf-idf score 0 will be missing
    
    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]
    corpus[i] = new_bow
    

[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 2), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 2), (18, 4), (19, 1)]


In [15]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus, 
                                           id2word=id2word,
                                           num_topics=3,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha="auto")

In [16]:
lda_model.print_topics()

[(0,
  '0.001*"indian" + 0.001*"man" + 0.001*"world" + 0.001*"big" + 0.001*"story" + 0.001*"beautiful" + 0.001*"believe" + 0.001*"feel" + 0.001*"religion" + 0.001*"thing"'),
 (1,
  '0.001*"man" + 0.001*"people" + 0.001*"world" + 0.001*"woman" + 0.001*"beef" + 0.001*"believe" + 0.001*"beautiful" + 0.001*"good" + 0.001*"indian" + 0.001*"leave"'),
 (2,
  '0.011*"man" + 0.008*"indian" + 0.008*"world" + 0.008*"believe" + 0.007*"story" + 0.007*"beautiful" + 0.007*"people" + 0.007*"feel" + 0.007*"good" + 0.007*"woman"')]

In [19]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word, mds="mmds", R=10)
vis

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2     -0.033117  0.032958       1        1  99.947747
1      0.016525 -0.016388       2        1   0.026366
0      0.016593 -0.016570       3        1   0.025887, topic_info=           Term       Freq      Total Category  logprob  loglift
583         man  27.000000  27.000000  Default  10.0000  10.0000
492      indian  20.000000  20.000000  Default   9.0000   9.0000
1047      world  19.000000  19.000000  Default   8.0000   8.0000
100     believe  18.000000  18.000000  Default   7.0000   7.0000
700      people  17.000000  17.000000  Default   6.0000   6.0000
...         ...        ...        ...      ...      ...      ...
91    beautiful   0.000677  17.776792   Topic3  -6.8472  -1.9168
634       movie   0.000667  12.518805   Topic3  -6.8624  -1.5813
352        feel   0.000674  16.904613   Topic3  -6.8519  -1.8711
944       thing   0.000670  15.148915   Topic3  -6.8568  -1.7664
100     believe   0.000675  18.652870   Topic3  -6.8505  -1.9681

[64 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
54        1  0.857054       arrive
91        1  1.012556    beautiful
97        1  0.990242         beef
100       1  1.018610      believe
102       1  0.990498          big
125       1  0.856846        breed
137       1  0.857111        burqa
144       1  0.856906       cannot
158       1  0.857213        chain
168       1  0.857416        choot
169       1  0.857496      chosque
170       1  0.857009     chrislam
265       1  0.857000       disney
280       1  0.857102      driving
352       1  1.005643         feel
416       1  1.005659         good
459       1  0.857011       hitler
481       1  0.856802      illegal
492       1  0.980907       indian
524       1  0.857499         keat
549       1  1.030250        leave
583       1  0.986211          man
603       1  0.856889  merchandise
634       1  1.038438        movie
700       1  1.012775       people
737       1  0.857295     probably
743       1  0.857253      pronoun
777       1  1.037767     religion
815       1  1.030124       school
884       1  0.856867       spread
895       1  0.856774        stick
900       1  1.012475        story
944       1  0.990170        thing
985       1  0.857190    twominute
992       1  0.857070    underwear
1004      1  0.856845     valuable
1042      1  1.006146        woman
1047      1  1.024657        world
1056      1  0.856750         yoga, R=10, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[3, 2, 1])

In [30]:
# Load JSON data from file
with open('vir_transcripts.json', 'r') as file:
    data = json.load(file)

# Extract transcripts from JSON data
transcripts = data.get('Transcripts', [])

# Iterate over each transcript
for transcript in transcripts:
    # Split the transcript into sentences
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', transcript)
    
    # Iterate over each sentence
    for sentence in sentences:
        # Check if 'friend' is in the sentence
        if 'woman' in sentence.lower():
            print(sentence)

My definition of feminism is not letting a woman be whatever a man can be.
It’s letting a woman be whatever a woman wants to be.
To limit… To limit a woman to the achievements of a man is to ask a scientist to become a monkey.
Every time a woman in India wears something revealing, like many of you are doing tonight, Indian men say shit like, 'Oh, she’s asking for it.' Am I wrong, ladies?
The infinite beauty and fun of being a woman is if she’s asking for it?
For a woman, the world is amazon.in.
That’s the beauty of being a woman.
